# SRΨ-v1.0 Baseline Training (Colab)

**独立子项目**: `srpsi-engine-tiny/colab_v1_baseline`

**目标**: 建立可靠的物理基线 (Energy Drift ~10.88)

**理论指导**: TRAE - "归一化即场标尺"

---

## Phase 1: 环境准备

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 安装依赖
!pip install torch torchvision numpy pyyaml tqdm tensorboard

In [ ]:
# 克隆代码库 (子项目)
import os

REPO_URL = 'https://github.com/Mozilla2004/srpsi-colab-baseline.git'  # TODO: 替换为真实 URL
SUBDIR = 'colab_v1_baseline'

# 方法 1: 克隆整个仓库，然后切换到子目录
if not os.path.exists('/content/srpsi-colab-baseline'):
    !git clone {REPO_URL} /content/srpsi-colab-baseline
    %cd /content/srpsi-colab-baseline/{SUBDIR}
else:
    %cd /content/srpsi-colab-baseline/{SUBDIR}

print("✅ 代码库已准备")
print(f"📁 当前目录: {os.getcwd()}")

## Phase 2: 数据准备

In [ ]:
# 生成数据
import numpy as np
import os
from datetime import datetime

# 创建输出目录
DATA_DIR = '/content/drive/MyDrive/srpsi-colab-baseline/data'
os.makedirs(DATA_DIR, exist_ok=True)

DATA_PATH = os.path.join(DATA_DIR, 'burgers_1d_v1.npy')

if not os.path.exists(DATA_PATH):
    print("📊 生成 Burgers 1D 数据...")
    
    from src.data_gen import generate_burgers_1d
    
    data = generate_burgers_1d(
        num_samples=4800,  # 4000 train + 400 val + 400 test
        total_steps=48,     # 16 tin + 32 tout
        nx=128,
        dt=0.01,
        dx=0.01,
        nu=0.01,
        seed=42
    )
    
    np.save(DATA_PATH, data)
    print(f"✅ 数据已保存: {DATA_PATH}")
    print(f"   Shape: {data.shape}")
else:
    print(f"✅ 数据已存在: {DATA_PATH}")
    data = np.load(DATA_PATH)
    print(f"   Shape: {data.shape}")

## Phase 3: 训练 SRΨ-v1.0

In [ ]:
# 配置训练
from datetime import datetime
import os

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 创建输出目录
OUTPUT_DIR = f'/content/drive/MyDrive/srpsi-colab-baseline/runs/v1_baseline_{timestamp}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 输出目录: {OUTPUT_DIR}")

# 训练命令
TRAIN_CMD = f"""
python src/train.py \
    --config config/burgers.yaml \
    --model srpsi_engine \
    --data {DATA_PATH} \
    --epochs 80 \
    --output {OUTPUT_DIR}
"""

print("🚀 开始训练...")
print(TRAIN_CMD)

In [ ]:
# 执行训练 (60-90 分钟)
!echo "$TRAIN_CMD" > train_cmd.sh
!bash train_cmd.sh

## Phase 4: 结果分析

In [ ]:
# 加载最佳模型并测试
import torch
from src.train import create_model, load_config, validate
from src.utils import get_device
from src.datasets import create_dataloaders

# 加载配置
cfg = load_config('config/burgers.yaml')
device = get_device('cuda')

# 创建模型
model = create_model('srpsi_engine', cfg, device)

# 加载 checkpoint
checkpoint_path = f"{OUTPUT_DIR}/checkpoints/final.pt"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✅ 模型已加载: {checkpoint_path}")

# 创建数据加载器
train_loader, val_loader, test_loader = create_dataloaders(
    data_path=DATA_PATH,
    tin=16,
    tout=32,
    batch_size=32,
    num_train=4000,
    num_val=400,
    num_test=400,
    num_workers=0
)

# 测试
print("\n🧪 运行测试...")
test_metrics = validate(model, test_loader, cfg, device)

print("\n" + "="*60)
print(" "*15 + "FINAL RESULTS")
print("="*60)
print(f"Test Loss:      {test_metrics['val_loss']:.6f}")
print(f"Test MSE:       {test_metrics['val_rollout_mse']:.6f}")
print(f"Test Drift:     {test_metrics['val_energy_drift']:.6f}")
print()

# 与 v1.0 基线对比
print("🎯 与 v1.0 基线对比:")
baseline_drift = 10.88
if test_metrics['val_energy_drift'] < baseline_drift:
    print(f"   ✅ Energy Drift 改善: {baseline_drift:.2f} → {test_metrics['val_energy_drift']:.2f}")
    print(f"   改善幅度: {((baseline_drift - test_metrics['val_energy_drift']) / baseline_drift * 100):.1f}%")
else:
    print(f"   Energy Drift: {test_metrics['val_energy_drift']:.2f} (基线: {baseline_drift:.2f})")

## Phase 5: 保存结果

In [ ]:
# 保存测试结果
import json
from datetime import datetime

results = {
    'model': 'SRΨ-v1.0 (Baseline)',
    'timestamp': datetime.now().isoformat(),
    'test_metrics': {
        'loss': float(test_metrics['val_loss']),
        'rollout_mse': float(test_metrics['val_rollout_mse']),
        'energy_drift': float(test_metrics['val_energy_drift'])
    },
    'baseline_comparison': {
        'v1_energy_drift': 10.88,
        'v1_momentum_drift': 18.49,
        'improvement': f"{((baseline_drift - test_metrics['val_energy_drift']) / baseline_drift * 100):.1f}%"
    },
    'trae_guidance': {
        'concept': 'Normalization as Field Ruler',
        'insight': 'Input standardization provides the geometric scale for field evolution'
    }
}

results_path = f"{OUTPUT_DIR}/test_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"💾 结果已保存: {results_path}")
print()
print("="*60)
print(" "*15 + "TRAINING COMPLETE ✅")
print("="*60)
print()
print("🎓 下一步: 分析 v1.0 的成功要素，准备注入 v2.0 特性")